# Data Transformation

- Loads raw PACMPL data
- Select columns, create dataframe, and save it.

In [2]:
import json
import pandas as pd
from pathlib import Path

In [3]:
DATA_RAW = Path("../data/raw/pacmpl_full.jsonl")

In [4]:
raw_data = []

with open(DATA_RAW) as f:
    for line in f:
        raw_data.append(json.loads(line))

print("Total PACMPL works:", len(raw_data))

Total PACMPL works: 2264


In [5]:
works_rows = []

for w in raw_data:

    referenced = [ref.split("/")[-1] for ref in w.get("referenced_works", [])  if ref]
    authors = []

    for a in w.get("authorships", []):
        author_obj = a.get("author", {})
        
        author_id_raw = author_obj.get("id")
        author_id = ( author_id_raw.split("/")[-1] if author_id_raw else None)

        orcid_raw = author_obj.get("orcid")
        orcid = (orcid_raw.split("/")[-1] if orcid_raw else None )

        authors.append({
            "author_id": author_id,
            "author_name": author_obj.get("display_name"),
            "orcid": orcid,
            "position": a.get("author_position"),
            "is_corresponding": a.get("is_corresponding")
        })

    works_rows.append({
        "work_id": w.get("id", "").split("/")[-1] if w.get("id") else None,
        "doi": w.get("doi"),
        "year": w.get("publication_year"),
        "title": w.get("title"),
        "type": w.get("type"),
        "volume": w.get("biblio", {}).get("volume"),
        "issue": w.get("biblio", {}).get("issue"),
        "referenced_works": referenced,
        "authorships": authors
    })

df_pacmpl = pd.DataFrame(works_rows)
df_pacmpl.head()

,work_id,doi,year,title,type,volume,issue,referenced_works,authorships
0,W2900153411,https://doi.org/10.1145/3290354,2019,An abstract domain for certifying neural networks,article,3,POPL,"[W178079818, W1883420340, W1932198206, W204487...","[{'author_id': 'A5100760604', 'author_name': '..."
1,W2590246587,https://doi.org/10.1145/3133901,2017,The tensor algebra compiler,article,1,OOPSLA,"[W60615445, W67471658, W202428374, W278615799,...","[{'author_id': 'A5041886781', 'author_name': '..."
2,W2898569715,https://doi.org/10.1145/3276486,2018,MadMax: surviving out-of-gas conditions in Eth...,article,2,OOPSLA,"[W10127936, W1536265389, W1993836075, W2014527...","[{'author_id': 'A5068595267', 'author_name': '..."
3,W2779850521,https://doi.org/10.1145/3158154,2017,RustBelt: securing the foundations of the Rust...,article,2,POPL,"[W75891272, W143008620, W173515685, W563467911...","[{'author_id': 'A5025527323', 'author_name': '..."
4,W2806718802,https://doi.org/10.1145/3276517,2018,DeepBugs: a learning approach to name-based bu...,article,2,OOPSLA,"[W202191487, W777621473, W1608271177, W1743635...","[{'author_id': 'A5013438083', 'author_name': '..."


In [6]:
OUTPUT_DIR = Path("../data/transformed")
file_path = OUTPUT_DIR / "pacmpl_master.parquet"
df_pacmpl.to_parquet(file_path, engine="pyarrow", index=False)